Due to the current situation (`Updated 09/02/2026`),

- Google Gemini has reduced the rate limits for several models, such as `gemini-2.5-flash` and `gemini-3-flash` (text models used in Colab notebooks), to a **limit of 20 Requests Per Day (RPD)**.

- To continue using these models seamlessly with sufficient rate limits, it is necessary to upgrade to the **pay-as-you-go tier** (link a Billing Account).
  - 👉 You can learn how to do this here: [https://ai.google.dev/gemini-api/docs/billing](https://ai.google.dev/gemini-api/docs/billing)

- Alternatively, you can follow the Groq API approach described below.

- Update `23/08/2026`: Removed the `llama` models because they are no longer supported by Groq, and replaced them with `qwen/qwen3.6-27b`.

- Update 26/09/2026: `qwen/qwen3.6-27b` is removed, change to `qwen3.8-27b`

Most of concepts and codes are adapted from this [repo](https://github.com/dair-ai/Prompt-Engineering-Guide).

Implementation Detail:
- LangChain is used.

# Setting environments and model setup

In [3]:
from IPython.display import display, Markdown

## Approach 1: Gemini

In [ ]:
# %%capture
# !pip install -qU langchain-google-genai

Request for Google API KEY here : https://aistudio.google.com/app/apikey

In [ ]:
# from getpass import getpass
# import os

# if "GOOGLE_API_KEY" not in os.environ:
#     os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google AI API key: ")

Since this is an open-ended generation, we do not want the generation to be boring so temperature is set to 1 instead of 0.

In [ ]:
# from langchain_google_genai import ChatGoogleGenerativeAI

# llm = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash",
#     temperature=1,
#     max_tokens=None,
#     timeout=None,
#     max_retries=2,
#     # other params...
# )

## Approach 2: Groq API


However, we still have an **alternative** that can be used via a free-tier API: **Groq API** (compatible with LangChain). This does not require linking a credit card and offers several models, such as:

Available models: https://console.groq.com/settings/limits

👉 You can sign up and get your API Key here: [https://console.groq.com/keys](https://console.groq.com/keys)


In [1]:
!pip install -qU langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.6 MB/s eta 0:00:00


In [2]:
import getpass
import os

os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


Since this is an open-ended generation, we do not want the generation to be boring so temperature is set to 1 instead of 0.

In [4]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="qwen/qwen3.8-27b",
    reasoning_effort="none",
    max_tokens=600,
    timeout=None,
    max_retries=2,
)

# Zero-shot / Few-shot Prompting

**Ex. Summarize the customer feedback.**

In [5]:
# Zero-shot

prompt = """Summarize the customer feedback.
Complaint:
The room was comfortable and had a good view.
However, the air conditioner was very noisy at night.
I contacted reception, but no technician arrived.
This made it difficult for me to sleep.
The hotel should respond to maintenance requests more quickly.
Summary:"""

zero_shot = llm.invoke(prompt)
display(Markdown(zero_shot.content))

The customer praised the room's comfort and view but complained about a noisy air conditioner that disturbed their sleep. Despite contacting reception, no technician arrived to fix the issue, leading to a request for faster response times to maintenance requests.

The output is in a free-form format. If you want the model to follow a specific structure, you can:
- Add output-format constraints directly to your prompt.
or
- Provide a few-shot example showing the desired format. (We will try this next).

In [6]:
# Few-shot

prompt = """Summarize the feedback following the example.

–Example 1–

Complaint:
The hotel room was clean, and the staff were friendly.
However, check-in took almost 40 minutes.
The room was also not ready at the promised time.
I had to wait in the lobby with my luggage.
The hotel should improve its check-in process.

Summary:
Positive: Clean room and friendly staff.

Issue: Long check-in time and delayed room availability.

Suggestion: Improve the check-in and room preparation process.

–Example 2–

Complaint:
The breakfast had a good variety of food.
However, several dishes were already cold.
The staff took a long time to refill empty items.
There were also not enough tables during busy hours.
The hotel should improve its breakfast service.

Summary:
Positive: Good variety of breakfast options.

Issue: Cold food, slow refills, and insufficient seating.

Suggestion: Improve food temperature, refill speed, and seating availability.

–Now–
Complaint:
The room was comfortable and had a good view.
However, the air conditioner was very noisy at night.
I contacted reception, but no technician arrived.
This made it difficult for me to sleep.
The hotel should respond to maintenance requests more quickly.
Summary:
"""

In [7]:
few_shot = llm.invoke(prompt)
display(Markdown(few_shot.content))

Summary:
Positive: Comfortable room with a good view.

Issue: Noisy air conditioner and lack of maintenance response.

Suggestion: Improve the speed of maintenance response to technical issues.

# Chain of Thought Prompting

**Example: Marketing Strategy Planning**

In [8]:
input_text = """
Should we spend the entire 500,000 THB marketing budget on Facebook Ads,
or should we distribute it across TikTok and Google Search as well?
Come up with an analysis and diversified plan suggestion.
"""

**1.) Without chain-of-thought**

In [9]:
prompt = f"""
{input_text}

Return the output within 200 words.
Explain in simple language.
"""

ai_msg_no_reasoning = llm.invoke(prompt)

display(Markdown("## Without Chain-of-Thought"))
display(Markdown(ai_msg_no_reasoning.content))

## Without Chain-of-Thought

Putting the entire 500,000 THB into Facebook is risky. It’s like betting all your chips on one card. While Facebook is powerful, relying on just one platform makes you vulnerable to algorithm changes and rising costs.

A diversified plan is smarter. Here’s a suggested split:

1.  **Facebook & Instagram (60% - 300,000 THB):** Use this for broad awareness and retargeting people who already know your brand. It’s great for keeping your business top-of-mind.
2.  **TikTok (25% - 125,000 THB):** This is crucial for reaching younger audiences. Short, creative videos can go viral, bringing in new customers who aren’t on Facebook.
3.  **Google Search (15% - 75,000 THB):** This captures high-intent users. When people actively search for what you sell, they are ready to buy. This channel often has the highest return on investment.

By splitting the budget, you reduce risk. If Facebook ads get expensive, TikTok might still perform well. If Google results are slow, Facebook keeps filling your funnel. This balanced approach ensures you reach customers at every stage: from discovering you on TikTok, to looking you up on Google, and finally making a purchase after seeing your Facebook ads. Diversification maximizes your total impact and protects your budget.

**Next, Zero-shot Chain-of-Thought :** `Let's think step-by-step`

In [10]:
prompt = f"""
{input_text}
Let's think step-by-step.
Return the output within 200 words.
Explain in simple language.
"""

ai_msg_no_reasoning = llm.invoke(prompt)

display(Markdown("## Zero-shot Chain-of-Thought"))
display(Markdown(ai_msg_no_reasoning.content))

## Zero-shot Chain-of-Thought

Putting all 500,000 THB into Facebook is risky. While Facebook has a huge audience, relying solely on it ignores where your customers actually are. Many Thai users spend significant time on TikTok for entertainment and Google for immediate needs. If you ignore these platforms, you miss out on high-intent buyers and younger demographics.

A diversified approach is safer and often more effective. Here is a suggested split:

1.  **Facebook & Instagram (50% - 250,000 THB):** Keep this as your core. It’s great for broad awareness and retargeting people who already know your brand.
2.  **TikTok (30% - 150,000 THB):** Use this for creative, viral-style content. It’s excellent for reaching younger audiences and generating buzz with less competition than Facebook.
3.  **Google Search (20% - 100,000 THB):** This captures high-intent users. When someone searches for your product, they are ready to buy. This ensures you don’t lose sales to competitors at the moment of decision.

Start by testing small budgets on all three. After two weeks, analyze which channel gives the lowest cost per acquisition (CPA) and highest return on ad spend (ROAS). Then, shift more budget to the winners. This strategy reduces risk and maximizes your reach across different customer journeys.

> Note: **Nowadays, many models use internal or implicit chain-of-thought reasoning by default**, so simply adding a ```zero-shot chain-of-thought``` prompt **may not** make a significant difference.

**2.) With your chain-of-thought**

> Thinking about :
- Product → Target Customer → Marketing Funnel → Budget Allocation → Risks → Final Recommendation

The answer will be improved.

In [11]:
prompt_reasoning = f"""
{input_text}

Before answering, reason through the following steps:

1. Identify the product/service and target customers.
   - If this information is unknown, assume two clearly different cases
     and provide separate recommendations.

2. Consider which stage of the marketing funnel each channel is best suited for:
   - Awareness
   - Consideration
   - Conversion

3. Estimate how the 500,000 THB budget could be allocated across channels.
   Consider whether each allocation is sufficient for test-and-learn.

4. Analyze the risks of each strategy.

5. Provide a final recommendation with an approximate budget allocation.

Return the output within 200 words.
Explain in simple language.
"""

ai_msg_reasoning = llm.invoke(prompt_reasoning)

display(Markdown("## With Chain-of-Thought"))
display(Markdown(ai_msg_reasoning.content))

## With Chain-of-Thought

**Case 1: Low-Consideration Products (e.g., Fast Fashion)**
Focus on impulse buys. Allocate 60% to TikTok for viral awareness, 30% to Facebook for retargeting, and 10% to Google for high-intent searches. This maximizes reach.

**Case 2: High-Consideration Services (e.g., Insurance)**
Focus on trust. Allocate 50% to Google Search to capture active intent, 30% to Facebook for educational ads, and 20% to TikTok for brand humanization.

**Analysis**
Each channel serves a funnel stage: TikTok drives awareness, Facebook aids consideration, and Google drives conversion. Spending 100% on Facebook is risky; it lacks top-of-funnel discovery and bottom-of-funnel intent capture.

**Budget Sufficiency**
With 500,000 THB, splitting into three channels allows ~150k–250k per channel. This is sufficient for initial testing. However, if your audience is niche, 150k might be too thin for meaningful data. In that case, stick to two channels (Facebook + Google) for deeper insights.

**Risks**
Single-channel strategies suffer from high customer acquisition costs over time due to saturation. Multi-channel risks spreading resources too thin, making it hard to optimize each platform effectively without dedicated management.

**Final Recommendation**
Adopt a diversified approach. For most businesses, a **40/30/30 split** (Facebook/TikTok/Google) balances discovery and conversion. This allows you to test creative and audience responses across different user mindsets, reducing reliance on a single algorithm and improving long-term ROI.

You can see this **difference** in the response:

`Since your product isn’t specified, let’s look at two common scenarios.`

- **Before CoT**: The **model mainly thinks about how to diversify** the budget.
- **After CoT**: The** model first considers the product **and target customer, **makes assumptions when needed**, then **analyzes the options** and gives a recommendation.

This **improves the depth and structure of the analysis**.

**3.) Few-shot Chain-of-Thought (Geometry Problem)**
> In cases where we want the reasoning steps to follow a clear structure — instead of letting the model generate its own free-form chain-of-thought — we can provide a few labeled examples to guide the reasoning process.
This technique is called few-shot chain-of-thought (few-shot CoT).

In [12]:
few_shot_cot_prompt = """Solve the following math problems step by step:

Example 1:
Problem: If a train travels at 60 miles per hour for 2.5 hours, how far does it go?
Solution:
1. Identify the given information:
   - Speed of the train: 60 miles per hour
   - Time of travel: 2.5 hours
2. Use the formula: Distance = Speed × Time
3. Plug in the values:
   Distance = 60 miles/hour × 2.5 hours
4. Calculate:
   Distance = 150 miles
Therefore, the train travels 150 miles.

Problem: A bakery sold 136 cakes last week. This week, they sold 25% more. How many cakes did they sell this week?
Solution:
1. Identify the given information:
   - Last week's sales: 136 cakes
   - Increase: 25%
2. Calculate the increase:
   25% of 136 = 0.25 × 136 = 34 cakes
3. Add the increase to last week's sales:
   This week's sales = 136 + 34 = 170 cakes
Therefore, the bakery sold 170 cakes this week.

Here is a new question:
Problem: If a rectangle has a length of 15 meters and a width of 8 meters, what is its area?
Solution:
"""

In [13]:
ans = llm.invoke(few_shot_cot_prompt).content
display(Markdown(ans))

Solution:
1. Identify the given information:
   - Length of the rectangle: 15 meters
   - Width of the rectangle: 8 meters
2. Use the formula: Area = Length × Width
3. Plug in the values:
   Area = 15 meters × 8 meters
4. Calculate:
   Area = 120 square meters
Therefore, the area of the rectangle is 120 square meters.

# Additional Topics:

## Analogy-based Prompting

Concept:
> **Let the model recall similar problems from other contexts or domains**, then **transfer useful patterns or lessons** to the current problem.

Expectation:
> **Quickly improve response quality—such as specificity, groundedness, or idea interestingness**—without requiring heavy prompt engineering, such as manually designing detailed reasoning steps or preparing few-shot examples.

**Ex. Employee Adoption of an Internal AI Tool**

In [14]:
problem = """
A company has introduced a new internal AI assistant for employees.

Most employees try the tool once or twice, but then stop using it.
The company has already provided basic training, but adoption remains low.

What should the company do to increase long-term usage?
"""

**Without analogy-based prompting**

In [15]:
prompt = f"""
Problem:
{problem}

Recommend a practical solution.
Keep the response concise.
"""

res = llm.invoke(prompt)

display(Markdown("## Without Analogy-Based Prompting"))
display(Markdown(res.content))

## Without Analogy-Based Prompting

To increase long-term adoption, the company should move from generic training to **workflow integration and gamification**.

1.  **Embed in Daily Workflows:** Integrate the AI directly into existing tools (e.g., email, Slack, or project management software) so it solves immediate, specific pain points rather than existing as a separate app.
2.  **Targeted "Champion" Strategy:** Identify and empower 5–10% of early-adopter employees as "AI Champions." Give them exclusive access, early features, or recognition. They will share practical tips and social proof with peers, which is more effective than top-down mandates.
3.  **Show Quick Wins:** Create simple, low-effort use cases (e.g., "Summarize this meeting notes" or "Draft this email") that deliver value in under 30 seconds.

**Core Principle:** Reduce friction and demonstrate immediate, tangible value in the user's existing routine.

**With analogy-based prompting**

In [16]:
prompt = f"""
Problem:
{problem}

Before solving the problem:

1. Recall 2 similar situations from other domains where people
   tried a new product or behavior but failed to continue using it.

2. Explain what helped improve long-term adoption in those situations.

3. Identify which lessons can be transferred to this workplace problem.

4. Apply those lessons to recommend a practical strategy
   for increasing long-term AI tool usage.

Keep the response concise.

Relevant analogies:

Final recommendation:
"""

ai_msg_with = llm.invoke(prompt)

display(Markdown("## With Analogy-Based Prompting"))
display(Markdown(ai_msg_with.content))

## With Analogy-Based Prompting

### Relevant analogies

1.  **Fitness Apps (e.g., Strava/MyFitnessPal)**
    *   *Situation:* Users download the app after a New Year’s resolution but stop after two weeks due to a lack of visible progress or social accountability.
    *   *Improvement:* Adoption increased when the apps introduced **social features** (challenges, leaderboards) and **gamification** (streaks, badges), turning a solitary chore into a shared, rewarding experience.

2.  **Language Learning Platforms (e.g., Duolingo)**
    *   *Situation:* Learners start with high motivation but quit when the content feels generic or disconnected from real-life needs.
    *   *Improvement:* Long-term retention improved through **micro-learning** (short, 5-minute lessons) and **personalized feedback** that adapted to the user’s specific pace and interests, reducing friction and maintaining momentum.

### Transferable Lessons

*   **Reduce Friction:** The tool must fit seamlessly into existing workflows rather than requiring a separate, dedicated "study" time.
*   **Social Proof & Accountability:** Individual usage is fragile; group dynamics and visibility drive consistency.
*   **Immediate Value:** Users need to see quick, tangible wins to justify the effort of continued use.

### Final recommendation

**Implement a "Workflow Integration & Peer Challenge" Strategy:**

1.  **Embed, Don’t Standalone:** Integrate the AI assistant directly into the company’s primary communication or project management tools (e.g., Slack, Teams, Jira) so it is accessible with one click during existing tasks, removing the need to "open a new tab."
2.  **Launch Micro-Challenges:** Create short, weekly team-based challenges (e.g., "Best AI-generated summary this week") with low-stakes recognition. This leverages social accountability and makes usage a collaborative activity rather than an individual burden.
3.  **Curate "Quick Wins" Templates:** Provide pre-built prompt templates for common, high-friction tasks (e.g., drafting emails, summarizing meeting notes) to ensure users experience immediate success within the first 30 seconds of use.

- **Without Analogy**: More generic and less interesting recommendations, based mainly on the immediate problem.
- **With Analogy**: Encourages the model to think through similar cases, which can lead to more interesting, specific, and grounded ideas.

Helpful Website for Prompting Techniques : https://www.promptingguide.ai/techniques

## LLM-as-Judge

**1. Prepare the ideas to evaluate.**

In [17]:
import pandas as pd
import json

# Ideas to evaluate
ideas = {
    "Idea 1": """
Introduce AI technology to automatically read and process
documents and forms across the organization.

The goal is to reduce manual data-entry workload,
improve overall operational speed, and significantly reduce costs.

The development team will immediately experiment with free
open-source tools so that the project can deliver results quickly.
""",

    "Idea 2": """
Develop an automated OCR system for processing invoice forms
for the Accounting Department.

The goal is to reduce manual data-entry time by 40%
and reduce the data error rate to below 2% within this quarter.

The project will run as a 3-week pilot using
1 developer and 1 accounting staff member
before full deployment.
"""
}

**2. Define the Evaluation Rubric**
> Using a detailed scoring rubric with clearly specified criteria is recommended to improve the consistency of the LLM judge.

In [18]:
rubric = """
### Clarity
- Score 1: Vague or confusing; no clear problem or target audience.
- Score 2: Broad idea, but lacks a defined problem and specific audience.
- Score 3: Understandable, but lacks important details or focus.
- Score 4: Clear problem and audience, with minor gaps in structure or focus.
- Score 5: Extremely clear, specific, and well-structured.

### Business Value
- Score 1: No clear business benefit.
- Score 2: Potential benefit, but weak connection to business goals.
- Score 3: Good potential impact, but lacks measurable KPIs.
- Score 4: Clear business alignment and measurable KPIs,
  but ROI or concrete impact is still incomplete.
- Score 5: Strong business alignment, measurable KPIs,
  and clear ROI or business impact.

### Feasibility
- Score 1: Unrealistic and lacks actionable steps.
- Score 2: Some steps are proposed, but major resource or execution issues remain.
- Score 3: Actionable, but missing important details such as who, when, or how.
- Score 4: Clear execution plan with minor gaps in resources or timeline.
- Score 5: Highly actionable with clear steps,
  realistic resources, ownership, and timeline.
"""

**3. Run the LLM-as-Judge Loop**

In [19]:
results = []

for idea_name, idea in ideas.items():

    prompt = f"""
You are a strict but fair Business Strategy Director.

Evaluate the following business idea using the rubric below.

{rubric}

Idea:
{idea}

Return ONLY valid JSON in this format:

{{
  "clarity": 1,
  "clarity_rationale": "...",
  "business_value": 1,
  "business_value_rationale": "...",
  "feasibility": 1,
  "feasibility_rationale": "...",
  "final_suggestion": "..."
}}
"""

    response = llm.invoke(prompt).content.strip()

    # Remove Markdown code fences if the model adds them
    response = response.replace("```json", "").replace("```", "").strip()

    result = json.loads(response)

    results.append({
        "Idea": idea_name,
        "Clarity": result.get("clarity"),
        "Business Value": result.get("business_value"),
        "Feasibility": result.get("feasibility"),
        "Clarity Rationale": result.get("clarity_rationale", ""),
        "Business Value Rationale": result.get("business_value_rationale", ""),
        "Feasibility Rationale": result.get("feasibility_rationale", ""),
        "Final Suggestion": result.get("final_suggestion", "N/A")
    })

df = pd.DataFrame(results)
display(df)

,Idea,Clarity,Business Value,Feasibility,Clarity Rationale,Business Value Rationale,Feasibility Rationale,Final Suggestion
0,Idea 1,3,3,3,The idea is understandable and identifies a ge...,There is good potential impact through cost re...,The plan is actionable in that it proposes usi...,Refine the scope to target a specific high-vol...
1,Idea 2,4,4,3,The problem (manual data entry) and target aud...,There is a clear alignment with business goals...,While the timeline (3 weeks) and resources (1 ...,N/A


**Designing LLM-as-Judge is iterative refinement work, you need to inspect the prompt <-> response of the judgement , calibrate the rubric to better align your task**



## Iterative Refinement

> Reflection Loop: We can integrate LLM-as-a-Judge into a refinement loop to automatically improve response quality based on the judge’s feedback.

Idea → Judge → Feedback → Refine → Re-evaluate → Compare Before vs. After

In [20]:
# Original low-scoring idea
idea_1 = """
Introduce AI technology to automatically read and process
documents and forms across the organization.

The goal is to reduce manual data-entry workload,
improve overall operational speed, and significantly reduce costs.

The development team will immediately experiment with free
open-source tools so that the project can deliver results quickly.
"""

**Detailed Rubric 1 to 5**

In [21]:
rubric = """
### Clarity
- Score 1: Vague or confusing; no clear problem or target audience.
- Score 2: Broad idea, but lacks a defined problem and specific audience.
- Score 3: Understandable, but lacks important details or focus.
- Score 4: Clear problem and audience, with minor gaps in structure or focus.
- Score 5: Extremely clear, specific, and well-structured.

### Business Value
- Score 1: No clear business benefit.
- Score 2: Potential benefit, but weak connection to business goals.
- Score 3: Good potential impact, but lacks measurable KPIs.
- Score 4: Clear business alignment and measurable KPIs,
  but ROI or concrete impact is still incomplete.
- Score 5: Strong business alignment, measurable KPIs,
  and clear ROI or business impact.

### Feasibility
- Score 1: Unrealistic and lacks actionable steps.
- Score 2: Some steps are proposed, but major resource or execution issues remain.
- Score 3: Actionable, but missing important details such as who, when, or how.
- Score 4: Clear execution plan with minor gaps in resources or timeline.
- Score 5: Highly actionable with clear steps,
  realistic resources, ownership, and timeline.
"""

**Judge Prompt Function**

In [22]:
def judge_idea(idea):

    prompt = f"""
You are a strict but fair Business Strategy Director.

Evaluate the following business idea using the rubric below.

{rubric}

Idea:
{idea}

Return ONLY valid JSON:

{{
  "clarity": 1,
  "business_value": 1,
  "feasibility": 1,
  "feedback": "Give concise and actionable feedback for improving the idea."
}}
"""

    response = llm.invoke(prompt).content.strip()

    response = (
        response
        .replace("```json", "")
        .replace("```", "")
        .strip()
    )

    return json.loads(response)

**Run the Reflection Loop**

In [23]:
from tqdm.notebook import tqdm

steps = [
    "Evaluate original idea",
    "Reflect and improve",
    "Evaluate improved idea"
]

with tqdm(total=len(steps), desc="Reflection Loop") as pbar:

    # Step 1: Evaluate the original idea
    before = judge_idea(idea_1)
    pbar.update(1)

    # Step 2: Improve the idea using judge feedback
    refinement_prompt = f"""
Original Idea:
{idea_1}

Evaluator Feedback:
{before["feedback"]}

Revise the idea to address the feedback.

Improve:
- clarity and scope
- measurable business value
- feasibility and execution details

Do not change the core objective of using AI for document processing.

Return only the improved idea.
"""

    improved_idea = llm.invoke(refinement_prompt).content.strip()
    pbar.update(1)

    # Step 3: Evaluate the improved idea
    after = judge_idea(improved_idea)
    pbar.update(1)

Reflection Loop:   0%|          | 0/3 [00:00<?, ?it/s]

**Shows the idea : before vs. after**

In [24]:
comparison = pd.DataFrame([
    {
        "Version": "Before",
        "Clarity": before["clarity"],
        "Business Value": before["business_value"],
        "Feasibility": before["feasibility"],
    },
    {
        "Version": "After",
        "Clarity": after["clarity"],
        "Business Value": after["business_value"],
        "Feasibility": after["feasibility"],
    }
])

comparison["Average"] = comparison[
    ["Clarity", "Business Value", "Feasibility"]
].mean(axis=1)

display(comparison)

,Version,Clarity,Business Value,Feasibility,Average
0,Before,2,3,2,2.333333
1,After,5,5,4,4.666667


In [25]:
# Show the ideas and judge feedback

display(Markdown("## Before Refinement"))
display(Markdown(idea_1))

display(Markdown("### Judge Feedback"))
display(Markdown(before["feedback"]))

## Before Refinement


Introduce AI technology to automatically read and process
documents and forms across the organization.

The goal is to reduce manual data-entry workload,
improve overall operational speed, and significantly reduce costs.

The development team will immediately experiment with free
open-source tools so that the project can deliver results quickly.


### Judge Feedback

The idea is too broad; define specific document types (e.g., invoices, contracts) and target departments. 'Immediately experimenting with free tools' lacks security, compliance, and integration considerations. Add specific KPIs (e.g., 50% reduction in processing time) and a phased rollout plan rather than an undefined 'immediate' start.

In [26]:
display(Markdown("## After Refinement"))
display(Markdown(improved_idea))

display(Markdown("### Judge Feedback"))
display(Markdown(after["feedback"]))

## After Refinement

**Project Title: Phased AI-Driven Document Automation for AP and Procurement**

**Objective**
Implement an AI-based Intelligent Document Processing (IDP) system to automate the extraction, validation, and routing of specific high-volume document types: **Vendor Invoices and Purchase Orders**. The primary target department is **Accounts Payable (AP)**, with a secondary extension to **Procurement** for PO matching.

**Scope & Specifics**
*   **Document Types:** Focus exclusively on standard vendor invoices and internal purchase orders in PDF and image formats.
*   **Target Departments:** Accounts Payable (primary), Procurement (secondary).
*   **Integration:** The system will integrate directly with the existing ERP (e.g., SAP/Oracle) via secure API endpoints to auto-populate fields, flag discrepancies, and trigger payment workflows.

**Execution Plan (Phased Rollout)**
*   **Phase 1: Security & Pilot (Months 1–3)**
    *   Conduct a security and compliance risk assessment to ensure data privacy (GDPR/CCPA) and access controls are met.
    *   Select a vetted, enterprise-grade IDP solution (evaluating open-source options like Tesseract/OCRmyPDF only if they meet strict security standards; otherwise, utilizing secure SaaS or on-premise commercial tools).
    *   Pilot with a subset of AP staff (10% of invoice volume) to test accuracy and integration stability.
*   **Phase 2: Optimization & Training (Months 4–5)**
    *   Refine AI models based on pilot feedback to achieve >95% data extraction accuracy.
    *   Train AP and Procurement teams on exception handling and system usage.
*   **Phase 3: Full Deployment (Month 6)**
    *   Roll out to 100% of AP invoice volume.
    *   Expand scope to include standard Purchase Order documents for three-way matching.

**Key Performance Indicators (KPIs)**
*   **Efficiency:** Achieve a **50% reduction** in average invoice processing time (from 4 hours to 2 hours per invoice).
*   **Cost:** Reduce manual data-entry labor costs by **30%** within the first year.
*   **Accuracy:** Maintain a data extraction accuracy rate of **>95%**, reducing manual rework and payment errors.
*   **Throughput:** Increase invoice processing capacity by **25%** without additional headcount.

**Feasibility & Risk Mitigation**
*   **Security:** All document processing will occur within a secure, encrypted environment. Open-source tools will only be used if they pass rigorous internal security audits; otherwise, compliant commercial solutions will be prioritized to avoid compliance gaps.
*   **Integration:** Pre-built connectors for major ERPs will be utilized to minimize custom development time

### Judge Feedback

Excellent structure and clear KPIs. To achieve a perfect score on feasibility, specify the estimated budget/cost range for the IDP solution and define the exact ownership (e.g., IT vs. Finance) for the security audit and integration phases. Additionally, clarify the fallback plan if the 95% accuracy target is not met in Phase 2.

Now, the LLM can automatically optimize the idea against the given rubric through a closed-loop refinement process.